In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG retail")
spark.sql("CREATE SCHEMA IF NOT EXISTS retail.silver")

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS retail.silver.customers (
    _extracted_at  TIMESTAMP,
    _source_table  STRING,
    city           STRING,
    country        STRING,
    created_at     TIMESTAMP,
    customer_id    STRING,
    email          STRING,
    first_name     STRING,
    is_deleted     BOOLEAN,
    last_name      STRING,
    segment        STRING,
    updated_at     TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS  retail.silver.orders (
    _extracted_at  TIMESTAMP,
    _source_table  STRING,
    created_at     TIMESTAMP,
    customer_id    STRING,
    is_deleted     BOOLEAN,
    order_id       STRING,    
    order_status   STRING,
    order_ts       TIMESTAMP,    
    shipping_city  STRING,
    total_amount   DOUBLE,      
    updated_at     TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail.silver.order_items (
    _extracted_at  TIMESTAMP,
    _source_table  STRING,
    created_at     TIMESTAMP,
    is_deleted     BOOLEAN,
    line_total     DOUBLE,
    order_id       STRING,
    order_item_id  STRING,
    product_id     STRING,
    quantity       BIGINT,
    unit_price     DOUBLE,
    updated_at     TIMESTAMP
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS retail.silver.products (
    _extracted_at  TIMESTAMP,
    _source_table  STRING,
    category       STRING,
    cost_price     DOUBLE,
    created_at     TIMESTAMP,
    is_deleted     BOOLEAN,
    name           STRING,
    product_id     STRING,
    subcategory    STRING,
    unit_price     DOUBLE,
    updated_at     TIMESTAMP
) USING DELTA
""")


print("✅ All silver tables created (or already exist)")

Reusable functions

In [0]:
def cast_boolean_columns(df, columns=["is_deleted"]):
    """Cast specified columns to boolean."""
    for col in columns:
        if col in df.columns:
            df = df.withColumn(col, F.col(col).cast("boolean"))
    return df


def deduplicate(df, primary_key):
    """
    Keep the latest record per primary_key.
    Tiebreaker: updated_at DESC, then _extracted_at DESC.
    """
    window = Window.partitionBy(primary_key).orderBy(
        F.col("updated_at").desc(),
        F.col("_extracted_at").desc()
    )
    return (
        df
        .withColumn("_rn", F.row_number().over(window))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )


def upsert_to_silver(df_source, silver_table, primary_key):
    """
    MERGE deduped source into silver table.
    - Match found + record is newer  → UPDATE all columns
    - No match + record is active    → INSERT
    - Existing records can become deleted via the update path
    """
    columns = df_source.columns
    col_mapping = {col: f"source.{col}" for col in columns}

    delta_table = DeltaTable.forName(spark, silver_table)

    (
        delta_table.alias("target")
        .merge(
            df_source.alias("source"),
            f"target.{primary_key} = source.{primary_key}"
        )
        .whenMatchedUpdate(
            condition="source.updated_at > target.updated_at",
            set=col_mapping
        )
        .whenNotMatchedInsert(
            condition="source.is_deleted = false",
            values=col_mapping
        )
        .execute()
    )


def validate_silver(silver_table, primary_key, key_columns):
    """
    Run quality checks on a silver table:
    1. Duplicates on primary key
    2. Nulls in key columns
    3. Record count summary (total / active / deleted)
    """
    df = spark.table(silver_table)
    table_name = silver_table.split(".")[-1].upper()

    print(f"\n{'='*50}")
    print(f" Validation — {table_name}")
    print(f"{'='*50}")

    # 1. Duplicates
    duplicates = df.groupBy(primary_key).count().filter("count > 1")
    dup_count = duplicates.count()
    status = "⚠️  WARNING: Duplicates found!" if dup_count > 0 else "✅ OK"
    print(f"Duplicates on {primary_key}: {dup_count} — {status}")
    if dup_count > 0:
        display(duplicates)

    # 2. Nulls
    for col in key_columns:
        null_count = df.filter(F.col(col).isNull()).count()
        status = "⚠️  WARNING" if null_count > 0 else "✅ OK"
        print(f"Nulls in {col}: {null_count} — {status}")

    # 3. Summary
    total    = df.count()
    deleted  = df.filter(F.col("is_deleted")).count()
    active   = total - deleted
    print(f"\nTotal records  : {total}")
    print(f"Active records : {active}")
    print(f"Deleted records: {deleted}")

    # 4. Sample
    print("\nSample (5 rows):")
    display(df.limit(5))


def process_silver_table(bronze_table, silver_table, primary_key, key_columns):
    """
    Full pipeline for one table:
    Read bronze → cast types → deduplicate → upsert → validate
    """
    print(f"\n▶ Processing {silver_table}...")

    # Read
    df_bronze = spark.table(bronze_table)

    # Cast
    df_cast = cast_boolean_columns(df_bronze)

    # Dedup
    df_deduped = deduplicate(df_cast, primary_key)
    print(f"  Bronze records : {df_bronze.count()}")
    print(f"  After dedup    : {df_deduped.count()}")

    # Upsert
    upsert_to_silver(df_deduped, silver_table, primary_key)
    print(f"  ✅ MERGE complete")

    # Validate
    validate_silver(silver_table, primary_key, key_columns)


print("✅ Functions loaded")

In [0]:
TABLES = [
    {
        "bronze_table" : "retail.bronze.customers",
        "silver_table" : "retail.silver.customers",
        "primary_key"  : "customer_id",
        "key_columns"  : ["customer_id", "email", "created_at"],
    },
    {
        "bronze_table" : "retail.bronze.orders",
        "silver_table" : "retail.silver.orders",
        "primary_key"  : "order_id",
        "key_columns"  : ["order_id", "customer_id", "order_ts", "order_status", "created_at"],
    },
    {
        "bronze_table" : "retail.bronze.order_items",
        "silver_table" : "retail.silver.order_items",
        "primary_key"  : "order_item_id",
        "key_columns"  : ["order_item_id", "order_id", "product_id", "quantity", "unit_price"],
    },
    {
        "bronze_table" : "retail.bronze.products",
        "silver_table" : "retail.silver.products",
        "primary_key"  : "product_id",
        "key_columns"  : ["product_id", "name", "cost_price", "unit_price", "category","subcategory", "created_at"],
    },
]

print(f"✅ Config loaded — {len(TABLES)} tables to process")

In [0]:
for table_config in TABLES:
    process_silver_table(**table_config)

print("\n✅ Silver layer pipeline complete")

In [0]:
spark.table("retail.silver.customers").printSchema()
spark.table("retail.silver.orders").printSchema()
spark.table("retail.silver.order_items").printSchema()
spark.table("retail.silver.products").printSchema()